# Phase 30 — the unboosted continuation

**No new idea. This banks a gain that has already been measured.**

The Phase 28 gate ran four arms of 1,500 steps from `ptify-16b-step6555.pth`.
Its **control** — `--soft-onset-boost 1.0`, which is bit-identical to plain
training — moved MAESTRO F1 **0.9400 → 0.9431** with pp misses **51.3% → 48.1%**.
That is ordinary continued training from a checkpoint that had not converged,
and it is currently unbanked.

## What this is NOT

**Not Phase 23.** That run passed `--loss-weights velocity=0.1` for 10,000
steps and **lost 0.0050 onset F1**, precision 0.8753 → 0.8657 — it invented
more notes, the opposite of what 16b's win was made of. The weights here are
left at their defaults, which is what the Phase 28 control actually ran.

**Not another soft-note attempt.** `--soft-onset-boost` is deliberately absent.
Phases 27 and 28 measured that route to a negative result twice over; see
`benchmarks/soft-onset-boost-README.md`.

## The honest risk

1,500 steps gained. **10,000 steps of the same recipe is not 6.7x the gain**,
and Phase 23 is the evidence: a long run from this checkpoint went backwards.
The difference was its loss weighting, but the possibility that this
checkpoint simply degrades with more training is real and untested. This run
is therefore **checkpointed every 30 minutes and scored at several step
counts**, not just at the end — so a peak part-way through is recoverable
rather than trained past.

In [ ]:
COMMIT = "phase-27-recall-diagnosis"   # branch, tag, or full SHA
REPO = "https://github.com/ImSe4n/PTify.git"

!pip install -q --no-deps git+{REPO}@{COMMIT}
!pip install -q --no-deps piano_transcription_inference torchlibrosa \
    mido pretty_midi librosa soundfile audioread soxr lazy_loader \
    msgpack mir_eval

In [ ]:
!git clone -q --depth 1 --branch {COMMIT} {REPO} /kaggle/working/PTify

import glob, os
print("--- attached datasets ---")
for path in sorted(glob.glob("/kaggle/input/*/")):
    print(path)
    for sub in sorted(glob.glob(path + "*"))[:6]:
        print("   ", os.path.basename(sub))

In [ ]:
# Paths verified against this account's mounts. The directory must CONTAIN the
# year folders (2004/ ... 2018/): the index stores relative paths.
AUDIO_ROOT = "/kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0"

# The DEPLOYABLE 16b file (172,037,521 bytes), not `step_6555.pt` (260,690,320).
# The larger one is the resumable training state -- same weights plus two Adam
# momentum buffers per parameter. `--init-checkpoint` accepts either shape, so
# the mistake does not error; it silently costs 88MB of upload.
INIT_CKPT = "/kaggle/input/datasets/imse4n/checkpoint/ptify-16b-step6555.pth"

import json, pathlib
index = json.load(open("/kaggle/working/PTify/benchmarks/maestro_segments.json"))
probe = pathlib.Path(AUDIO_ROOT) / index["tracks"][0]["audio_filename"]
print("probe :", probe)
print("EXISTS:", probe.exists(), "<- must be True")
print("init  :", pathlib.Path(INIT_CKPT).exists(), "<- must be True")

## Train

`--resume auto` starts fresh when there is no checkpoint and continues when
there is, so **re-running this cell after a session dies is the recovery
procedure**.

`--keep-checkpoints 6` rather than 3: this run is scored at SEVERAL step
counts, so intermediate states have to survive. Phase 23 went backwards over
10,000 steps and only its endpoint was kept, which made "where did it peak?"
unanswerable without re-running the whole thing.

**No `--loss-weights` and no `--soft-onset-boost`.** Both defaults are
load-bearing here — see the intro.

In [ ]:
!cd /kaggle/working/PTify && python -m training.train \
    --index benchmarks/maestro_segments.json \
    --audio-root {AUDIO_ROOT} \
    --out /kaggle/working/checkpoints \
    --init-checkpoint {INIT_CKPT} \
    --augment \
    --augment-seed 0 \
    --augment-clean-prob 0.10 \
    --device cuda \
    --no-amp \
    --steps 6000 \
    --batch-size 4 \
    --accum-steps 2 \
    --workers 2 \
    --log-every 50 \
    --validate-every 250 \
    --val-batches 20 \
    --save-every-seconds 1800 \
    --save-every-steps 1500 \
    --keep-checkpoints 6 \
    --resume auto

In [ ]:
# Validation is scored UNWEIGHTED, so these are comparable across runs and
# against the Phase 28 control's 0.0078.
import json, pathlib

rows = [json.loads(l) for l in
        pathlib.Path("/kaggle/working/checkpoints/train_log.jsonl").open()]
vals = [r for r in rows if "val_onset" in r]

print(f"{'step':>6} {'val_total':>10} {'val_onset':>10}")
for r in vals:
    print(f"{r['step']:>6} {r['val_total']:>10.4f} {r['val_onset']:>10.5f}")

if vals:
    best = min(vals, key=lambda r: r["val_onset"])
    print(f"\nlowest val_onset at step {best['step']}: {best['val_onset']:.5f}")
    print("Phase 28 control, 1500 steps, was 0.0078 -- a run that does not beat")
    print("that has trained past its own peak, which is what Phase 23 did.")

In [ ]:
# Name the deployable file by STEP COUNT. Every arm of the Phase 28 gate wrote
# `ptify-note-pedal.pth`, the browser numbered them (1)(2)(3) on download by
# download order rather than arm order, and the deployable checkpoint stores
# only ['model'] -- so which file was which had to be recovered afterwards by
# reproducing a loss ordering. Do not repeat that.
import shutil, pathlib

src = pathlib.Path("/kaggle/working/checkpoints/ptify-note-pedal.pth")
if src.exists():
    dst = pathlib.Path("/kaggle/working/ptify-continuation-6000.pth")
    shutil.copy(src, dst)
    print(f"{dst.name}: {dst.stat().st_size / 1e6:.1f} MB")

import sys
sys.path.insert(0, "/kaggle/working/PTify")
from training.model import assert_deployable

assert_deployable(str(src))
print("verified deployable")

## Afterwards, on the laptop — IN THIS ORDER

```bash
# 1. RECALIBRATE FIRST. A retrained onset head invalidates the 0.6 threshold,
#    and scoring through a stale one misattributes a decode artifact to the
#    weights -- the mistake Phase 19 undid and Phase 23 nearly repeated.
set PTIFY_CHECKPOINT=C:\path\to\ptify-continuation-6000.pth
python -m tools.calibrate_thresholds --audio-dir recordings/maps_paired \
    --engine ptify --limit 6

# 2. Score at the RECALIBRATED threshold against the controlled baseline,
#    benchmarks/real/maps-paired-ptify16b-at060.json (0.8502).
#    NEVER compare against a report scored at a different threshold: that
#    confound produced Phase 23's phantom +0.0057 (HANDOFF section 1a).

# 3. Re-run the Phase 27 diagnosis, which is what says whether the gain is
#    real recall or just more invented notes.
python -m tools.recall_diagnosis --audio-dir recordings/maestro_test12 \
    --engine ptify --json benchmarks/recall-diagnosis-phase30.json
```

**Ship it only if `onset_p` holds.** 16b's entire published gain was a 37% cut
in invented notes, and a continuation that raises recall by inventing more is
the Phase 23 failure repeated. Recall alone is not success.